In [2]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import xarray as xr
from glob import glob
import os
from netCDF4 import Dataset
import pandas as pd
from datetime import datetime, date, timedelta
from pathlib import Path
import scipy
import scipy.ndimage
from mpl_toolkits.axes_grid1 import ImageGrid
import math
import cc3d

from mpl_toolkits.axes_grid1 import make_axes_locatable

#constants
t = 6 # time index
z_min = 0
z_max = 200

#Input
input_dir = Path("/mnt/stor-pool-01/projects/heus/ShellAnalysis/full-area/")
source_input_dir = Path("/mnt/stor-pool-01/projects/heus/EUREC4A_Eulerian/Feb_1st_12day_cdnc70_nudge/")

In [3]:
#load datasets
ds_ql_mask = xr.open_dataset(input_dir / "ql_mask.nc", decode_times=False)
ds_w_mask = xr.open_dataset(input_dir / "w_mask.nc", decode_times=False)
ds_shell_mask = xr.open_dataset(input_dir / "shell_mask.nc", decode_times=False)
ds_shell_labels = xr.open_dataset(input_dir / "shell_labels.nc", decode_times=False)
ds_cloud_labels = xr.open_dataset(input_dir / "cloud_labels.nc", decode_times=False)

ds_ql = xr.open_dataset(source_input_dir / "ql.nc", decode_times=False,chunks={'time': 1})
ds_w = xr.open_dataset(source_input_dir / "w.nc", decode_times=False,chunks={'time': 1})
ds_w = ds_w.rename({'zh':'z'}).interp(z=ds_ql.z)

In [4]:
shell_labels_slice_z_range = ds_shell_labels.isel(time=t).sel(z=slice(z_min, z_max))
low_shell_ids = np.unique(shell_labels_slice_z_range.shell_labels.where(shell_labels_slice_z_range.shell_labels > 0).values)

shell_mask_slice = ds_shell_mask.isel(time=t)
cloud_mask_slice = ds_ql_mask.isel(time=t)
shell_labels_slice = ds_shell_labels.isel(time=t)
cloud_labels_slice = ds_cloud_labels.isel(time=t)

In [5]:
base_full_shell = shell_mask_slice["shell_mask"].where(shell_mask_slice["shell_mask"] != 0).values
base_full_cloud = cloud_mask_slice["ql_mask"].where(cloud_mask_slice["ql_mask"] != 0).values

#extract labels
shell_labels_3d_slice = shell_labels_slice.shell_labels.values[:, :, :]
cloud_labels_3d_slice = cloud_labels_slice.cloud_labels.values[:, :, :]

In [6]:
first_regular = -1
first_ql_dilation = -1
first_periodic = -1
first_either = -1
first_dilated_and_periodic = -1

i = 0
while (first_periodic == -1 or first_ql_dilation == -1 or first_dilated_and_periodic == -1 or first_regular == -1) and i < len(low_shell_ids):
    target_id = low_shell_ids[i]
    print(f"Testing index = {i} (ID: {target_id}): ", end="")
    regular_test_passed = False
    regular_empty = False
    fragmented = False
    dilated_test_passed = False
    periodic_test_passed = False
    d_and_p_combined_test_passed = False    


    # --- Assembling Variables ---
    depth_3d_full_shell = np.where(shell_labels_3d_slice == target_id, base_full_shell, np.nan)
    depth_3d_full_cloud = np.where(cloud_labels_3d_slice == target_id, base_full_cloud, np.nan)
    is_shell = ~np.isnan(depth_3d_full_shell)
    is_cloud = ~np.isnan(depth_3d_full_cloud)
    merged_np = is_shell | is_cloud
    padded_merged_np = np.pad(merged_np, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)

    #assemble dilated version
    expansion = np.zeros((3,3,3), dtype=bool)
    expansion[1, 1, :] = True  # X axis
    expansion[1, :, 1] = True  # Y axis
    expansion[:, 1, 1] = True  # Z axis

    dilated_is_cloud = np.pad(is_cloud, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)
    dilated_is_cloud = np.pad(dilated_is_cloud, ((0, 0), (1, 1), (1, 1)), mode='wrap')
    dilated_is_cloud = scipy.ndimage.binary_dilation(dilated_is_cloud, structure=expansion, iterations=1)
    dilated_is_cloud = dilated_is_cloud[1:-1, 1:-1, 1:-1]
    dilated_merged_np = is_shell | dilated_is_cloud
    padded_d_merged_np = np.pad(dilated_merged_np, ((1, 1), (0, 0), (0, 0)), mode='constant', constant_values=0)


    # --- Testing ---
    # Test 1: no dilation and non periodic
    labeled_array, num_features = scipy.ndimage.label(merged_np)
    if num_features == 1:
        regular_test_passed = True
    elif num_features == 0:
        regular_empty = True

    # Test 2: periodic
    padded_labeled_array = cc3d.connected_components(
        padded_merged_np, 
        connectivity=6, 
        periodic_boundary=True
    )
    labeled_array = padded_labeled_array[1:-1, :, :].astype(np.uint32)

    num_features = int(labeled_array.max())

    if num_features == 1:
        periodic_test_passed = True

    # Test 3: dilated ql
    labeled_array, num_features = scipy.ndimage.label(dilated_merged_np)

    if num_features == 1:
        dilated_test_passed = True

    # Test 4: dilated ql and periodic
    padded_labeled_array = cc3d.connected_components(
        padded_d_merged_np, 
        connectivity=6, 
        periodic_boundary=True
    )
    labeled_array = padded_labeled_array[1:-1, :, :].astype(np.uint32)

    num_features = int(labeled_array.max())

    if num_features == 1:
        d_and_p_combined_test_passed = True
    elif num_features > 1:
        fragmented = True

    # --- Evaluation ---
    must_be_periodic = (not regular_test_passed) and periodic_test_passed
    must_be_dilated =  (not regular_test_passed) and dilated_test_passed
    must_be_d_and_p = (not (regular_test_passed or periodic_test_passed or dilated_test_passed)) and d_and_p_combined_test_passed

    #printed statements
    if must_be_d_and_p:
        print("must be both dilated and periodic")
    elif must_be_periodic and (not must_be_dilated):
        print("must be periodic")
    elif must_be_dilated and (not must_be_periodic):
        print("must be dilated")
    elif must_be_dilated and must_be_periodic:
        print("must be either dilatied or periodic")
    elif regular_test_passed:
        print("does not need to be dilated or periodic")
    elif regular_empty:
        print("ERROR: cloud and shell is empty")
    elif fragmented:
        print("ERROR: fragmented even when periodic and dilated")
    else:
        print("ERROR: Unknown issue")

    #variable updates
    if first_regular == -1 and regular_test_passed:
        first_regular = i
    if first_periodic == -1 and must_be_periodic:
        first_periodic = i
    if first_ql_dilation == -1 and must_be_dilated:
        first_ql_dilation = i
    if first_either == -1 and must_be_periodic and must_be_dilated:
        first_either = i
    if first_dilated_and_periodic == -1 and must_be_d_and_p:
        first_dilated_and_periodic = i

    i += 1
print("Testing complete\n")

print("-" * 50)
print(f"RESULTS FOR TIMESTEP {t}")
print("-" * 50)
print("\n")
if first_regular != -1:
    print(f"Index of first regular test passed: {first_regular}")
else:
    print("No indexes were found that passed the regular test")

if first_periodic != -1:
    print(f"Index of first one that needed to be periodic: {first_periodic}")
else:
    print("No indexes were found that needed to be periodic")

if first_ql_dilation != -1:
    print(f"Index of first one that needed to have ql dilated: {first_ql_dilation}")
else: 
    print("No indexes were found that needed ql dilated")

if first_either != -1:
    print(f"[STRANGE CASE] index of first one that needs either ql dilation or to be periodic: {first_either}")

if first_dilated_and_periodic != -1:
    print(f"Index of first one that needed to have both ql dilated and be periodic: {first_dilated_and_periodic}")
else:
    print("No indexes were found that needed to be both periodic and have ql dilated")



    


    
    


Testing index = 0 (ID: 2.0): does not need to be dilated or periodic
Testing index = 1 (ID: 3.0): must be both dilated and periodic
Testing index = 2 (ID: 4.0): must be dilated
Testing index = 3 (ID: 5.0): does not need to be dilated or periodic
Testing index = 4 (ID: 6.0): does not need to be dilated or periodic
Testing index = 5 (ID: 7.0): does not need to be dilated or periodic
Testing index = 6 (ID: 8.0): does not need to be dilated or periodic
Testing index = 7 (ID: 9.0): does not need to be dilated or periodic
Testing index = 8 (ID: 10.0): must be dilated
Testing index = 9 (ID: 11.0): does not need to be dilated or periodic
Testing index = 10 (ID: 12.0): must be dilated
Testing index = 11 (ID: 13.0): must be dilated
Testing index = 12 (ID: 15.0): does not need to be dilated or periodic
Testing index = 13 (ID: 16.0): does not need to be dilated or periodic
Testing index = 14 (ID: 17.0): does not need to be dilated or periodic
Testing index = 15 (ID: 18.0): does not need to be dila